# Data Familiarization & Preprocessing & Exploration:

#### **Checklist**

* [ ] List variables, formats, units for each dataset
* [ ] Preview and inspect data
* [ ] Standardize all units and timestamps
* [ ] Filter to region and period of interest
* [ ] Handle missing and erroneous data
* [ ] Merge datasets (if needed)
* [ ] Summarize and visualize key variables
* [ ] Document everything

## GPM IMERG Data

In [ ]:
pip install rasterio

In [ ]:
pip install matplotlib

In [ ]:
import rasterio # Like terra in R
import matplotlib.pyplot as plt

# Open the file
with rasterio.open(r'../Data/IMERG/gpm_aoi.tiff') as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Width, Height:", src.width, src.height)
    print("Number of bands:", src.count)
    
    # Read the first band (most common for single-variable rasters)
    data = src.read(1)
    print("Data shape:", data.shape)
    print("Min, Max:", data.min(), data.max())

    # Quick plot
    plt.imshow(data, cmap='viridis')
    plt.colorbar(label='Value')
    plt.title('gpm_aoi.tiff')
    plt.show()

In [ ]:
pip install pandas pyarrow

In [ ]:
import pandas as pd

# Read the parquet file
imgerg_df = pd.read_parquet(r'..\Data\IMERG\imerg_data-20250731T220028Z-1-001\imerg_data\gpm_imerg_2024-10-11.parquet')

# Preview the data
print(imgerg_df.shape)        
print(imgerg_df.columns)       
print(imgerg_df.head())       

# For summary statistics
print(imgerg_df.describe())

# For info on dtypes and missing values
print(imgerg_df.info())


In [ ]:
imgerg_df.head(10)

**The "probabilityLiquidPrecipitation" field is a probability from 0 to 100, 0% = MOST likely snow and 100% = MOST likely rain.**

In [ ]:
# Map probabilityLiquidPrecipitation for a given time:
import matplotlib.pyplot as plt
plt.scatter(imgerg_df['x'], imgerg_df['y'], c=imgerg_df['2024-10-11 14:00:00'], cmap='coolwarm', s=20)
plt.colorbar(label='% Probability Rain')
plt.title('GPM IMERG Prob. Liquid Precipitation at 14:00')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.show()

In [ ]:
# Pick a location (row) and plot its probability vs. time:
row = imgerg_df.iloc[0]
time_cols = imgerg_df.columns[2:]  # skip x, y
plt.plot(time_cols, row[time_cols])
plt.xticks(rotation=90)
plt.title(f"Time Series at x={row['x']}, y={row['y']}")
plt.ylabel('% Probability Rain')
plt.show()

In [ ]:
# Average over all locations to see domain-wide mean over time:
means = imgerg_df[time_cols].mean(axis=0)
plt.plot(time_cols, means)
plt.xticks(rotation=90)
plt.title("Domain-Averaged GPM IMERG Probability Over Time")
plt.ylabel('% Probability Rain')
plt.show()

In [ ]:
# What’s the min, max, mean, and distribution for each time? For each location?
imgerg_df[time_cols].describe()

In [ ]:
# Which grid points are most “rainy” (highest mean probability over the day)? Most “snowy” (lowest mean)?
imgerg_df['mean_prob'] = imgerg_df[time_cols].mean(axis=1)
imgerg_df[['x', 'y', 'mean_prob']].sort_values('mean_prob', ascending=False).head()

In [ ]:
# Classify each value as "rain" (prob > 50), "snow" (prob < 50), or "mixed" (close to 50):
import numpy as np
rain_map = (imgerg_df[time_cols] > 50).astype(int)  # 1 = rain, 0 = snow
# Or, count how often each cell was rain, snow, or mixed
rain_counts = (imgerg_df[time_cols] > 50).sum(axis=1)
snow_counts = (imgerg_df[time_cols] < 50).sum(axis=1)

print (f"rain counts: {rain_counts}\n\nsnow counts: {snow_counts}")

In [ ]:
# Find locations with the highest variability (std across time):
imgerg_df['std_prob'] = imgerg_df[time_cols].std(axis=1)
imgerg_df

In [ ]:
# Heatmap for a Single Time Step
import matplotlib.pyplot as plt

# Pick a time column to visualize (change as needed)
time_col = '2024-10-11 14:00:00'

plt.figure(figsize=(8, 6))
plt.scatter(imgerg_df['x'], imgerg_df['y'], c=imgerg_df[time_col], cmap='coolwarm', s=30, vmin=0, vmax=100)
plt.colorbar(label='% Probability Rain')
plt.title(f'GPM IMERG Prob. Liquid Precipitation at {time_col}')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()


In [ ]:
# Animation of Heatmaps Over Time
%matplotlib inline
from IPython.display import HTML

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

time_cols = df.columns[2:]
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df['x'], df['y'], c=df[time_cols[0]], cmap='coolwarm', s=30, vmin=0, vmax=100)
plt.colorbar(sc, ax=ax, label='% Probability Rain')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
title = ax.set_title(f'Time: {time_cols[0]}')

def animate(i):
    sc.set_array(df[time_cols[i]])
    title.set_text(f'Time: {time_cols[i]}')
    return sc, title

anim = FuncAnimation(fig, animate, frames=len(time_cols), interval=200, blit=False)

# This line actually shows the animation in the notebook
HTML(anim.to_jshtml())


### Load and Combine All IMERG Daily Files

In [ ]:
# Find All Files
import glob
import pandas as pd

# Find all CSV files in the folder (change to *.parquet for parquet files)
file_list = glob.glob(r'..\Data\IMERG\imerg_data-20250731T220028Z-1-001\imerg_data\*.parquet')
print("nume of files: ", len(file_list))
print("Found files:", file_list)

In [ ]:
# For testing
file_list = file_list[:5]

In [ ]:
# Load Compine all IMERG file & Add a "Date" Identifier
import pandas as pd

all_dfs = []

for file in file_list:
    df = pd.read_parquet(file)

    # Extract the date from filename
    date_str = file.split('_')[-1].split('.')[0]  # e.g., '2024-10-11'

    # First two columns are assumed to be spatial (e.g., x and y)
    df = df.rename(columns={0: 'x', 1: 'y'})

    # Rename time columns to just HH:MM
    time_cols = df.columns[2:]
    new_time_cols = [pd.to_datetime(col).strftime('%H:%M') for col in time_cols]
    df.columns = ['x', 'y'] + new_time_cols

    # Add date column
    df['date'] = date_str

    # Reorder columns: x, y, date, time1, time2, ...
    cols = ['x', 'y', 'date'] + new_time_cols
    df = df[cols]

    all_dfs.append(df)

# Combine all DataFrames
big_df = pd.concat(all_dfs, ignore_index=True)

print(big_df.shape)


In [ ]:
big_df

In [ ]:
big_df.info()

In [ ]:
# Distribution of probability values:
time_cols = big_df.columns[3:]  # Skip x, y, date
all_probs = big_df[time_cols].values.flatten()

import matplotlib.pyplot as plt
plt.hist(all_probs, bins=40)
plt.title("Distribution of All ProbabilityLiquidPrecipitation Values")
plt.xlabel("% Probability")
plt.ylabel("Count")
plt.show()

In [ ]:
# Mean Probability by Hour of Day
# Calculate the mean value for each half-hour across all locations and dates:
mean_by_time = big_df[time_cols].mean(axis=0)

plt.plot(time_cols, mean_by_time)
plt.xticks(rotation=90)
plt.title("Mean Probability by Time of Day")
plt.ylabel("% Probability")
plt.xlabel("Time")
plt.show()

In [ ]:
# Mean Probability by Date
# Calculate the mean value for each date (across all grid cells and time steps):
big_df['mean_prob_by_date'] = big_df[time_cols].mean(axis=1)
mean_by_date = big_df.groupby('date')['mean_prob_by_date'].mean()

mean_by_date.plot(kind='bar')
plt.title("Domain-Averaged Probability by Date")
plt.ylabel("% Probability")
plt.xlabel("Date")
plt.show()

In [ ]:
# Cells With Most Frequent "Rain" or "Snow"
# Classify as "rain" (>50) or "snow" (<=50), and count per cell (row):

rain_counts = (big_df[time_cols] > 50).sum(axis=1)
snow_counts = (big_df[time_cols] <= 50).sum(axis=1)

# Add to df if you like
big_df['rain_hours'] = rain_counts/2
big_df['snow_hours'] = snow_counts/2
print("Cell with most rain hours:")
big_df[['x', 'y', 'date', 'rain_hours', 'snow_hours']].sort_values('rain_hours', ascending=False).head()


In [ ]:
# Missing Data Check

In [ ]:
missing = big_df[time_cols].isnull().sum().sum()
print(f"Total missing values in all probability fields: {missing}")

In [ ]:
# Animation of Heatmaps Over Time
import numpy as np

# Get unique dates and time columns
all_dates = big_df['date'].unique()
time_cols = big_df.columns[3:]

# Prepare a list of (date, time_col) for animation frames
frames = []
for date in all_dates:
    for time_col in time_cols:
        frames.append((date, time_col))

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

mpl.rcParams['animation.embed_limit'] = 100

fig, ax = plt.subplots(figsize=(8, 6))

# Start with the first frame's data
first_date, first_time = frames[0]
df_frame = big_df[big_df['date'] == first_date]
sc = ax.scatter(df_frame['x'], df_frame['y'], c=df_frame[first_time], cmap='coolwarm', s=30, vmin=0, vmax=100)
plt.colorbar(sc, ax=ax, label='% Probability Rain')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
title = ax.set_title(f'{first_date} — {first_time}')

def animate(i):
    date, time_col = frames[i]
    df_frame = big_df[big_df['date'] == date]
    sc.set_array(df_frame[time_col].values)
    title.set_text(f'{date} — {time_col}')
    return sc, title

anim = FuncAnimation(fig, animate, frames=len(frames), interval=200, blit=False)

HTML(anim.to_jshtml())

In [ ]:
# an Interactive Map
import folium
import pandas as pd
import numpy as np

big_df['x'] = pd.to_numeric(big_df['x'], errors='coerce')
big_df['y'] = pd.to_numeric(big_df['y'], errors='coerce')

chosen_date = '2024-10-13'
mask = big_df['date'] == chosen_date
filtered = big_df.loc[mask].dropna(subset=['x', 'y'])

print(f"Rows for chosen date: {filtered.shape[0]}")

center_lat = filtered['y'].mean()
center_lon = filtered['x'].mean()

import folium
import numpy as np

m = folium.Map(location=[center_lat, center_lon], zoom_start=7)

for _, row in filtered.iterrows():
    value = row['14:00']  # or any chosen time
    # Use a simple color scale for demo (blue for snow, red for rain)
    color = 'blue' if value < 50 else 'red'
    folium.CircleMarker(
        location=[row['y'], row['x']],
        radius=5,
        popup=f"Prob: {value:.1f}%",
        color=color,
        fill=True,
        fill_opacity=0.8
    ).add_to(m)

m


In [ ]:
pip install ipywidgets

In [ ]:
# an interactive map with a widget to choose the date (ZA: ~8 mintues to run on my local machine)
import ipywidgets as widgets
from IPython.display import display

def interactive_folium(date, time_col):
    mask = big_df['date'] == date
    filtered = big_df.loc[mask].dropna(subset=['x', 'y'])
    center_lat = filtered['y'].mean()
    center_lon = filtered['x'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=7)
    for _, row in filtered.iterrows():
        value = row[time_col]
        color = 'blue' if value < 50 else 'red'
        folium.CircleMarker(
            location=[row['y'], row['x']],
            radius=5,
            popup=f"Prob: {value:.1f}%",
            color=color,
            fill=True,
            fill_opacity=0.8
        ).add_to(m)
    display(m)

date_widget = widgets.Dropdown(options=sorted(big_df['date'].unique()), description='Date:')
time_widget = widgets.Dropdown(options=big_df.columns[3:], description='Time:')

widgets.interactive(interactive_folium, date=date_widget, time_col=time_widget)


In [ ]:
# Save the map
for date in big_df['date'].unique():
    m = plot_imerg_map_for_date_time(big_df, date, '14:00')
    m.save(f"imerg_{date}_14-00.html")

## MRoS Observations Data

In [ ]:
import pandas as pd

# Read the parquet file
mros_df = pd.read_parquet(r'..\Data\observations\wy25_mros_obs.parquet')

# Preview the data
print(mros_df.shape)         # (rows, columns)
print(mros_df.columns)       # column names
print(mros_df.head())        # first 5 rows

# For summary statistics
print(mros_df.describe())

# For info on dtypes and missing values
print(mros_df.info())


In [ ]:
mros_df

In [ ]:
# How many Rain vs Snow reports? Are there rare types?
mros_df['phase'].value_counts()

In [ ]:
# check for missing data:
mros_df.isnull().sum()


In [ ]:
# Unique value counts
mros_df.nunique()


In [ ]:
# How many observations per phase over time?

mros_df['date'] = pd.to_datetime(mros_df['createdtime']).dt.date
mros_df.groupby(['date', 'phase']).size().unstack().plot(kind='line', title="Observations per phase over time")


In [ ]:
pip install folium

In [ ]:
# Map of submissions
import folium
from folium.plugins import HeatMap

m = folium.Map(location=[39.5, -98.35], zoom_start=4)  # Center of USA
HeatMap(mros_df[['latitude', 'longitude']].dropna().values).add_to(m)
m

In [ ]:
# how many records are exact duplicates:
mros_df.duplicated(subset=['latitude', 'longitude', 'timestamp']).sum()

In [ ]:
# how many records are from the same lat & long:
mros_df.duplicated(subset=['latitude', 'longitude']).sum()


In [ ]:
# Compare createdtime, time_submitted_utc, and datetime_received_pacific to check latency or delays:
mros_df['createdtime'] = pd.to_datetime(mros_df['createdtime'])
mros_df['datetime_received_pacific'] = pd.to_datetime(mros_df['datetime_received_pacific'])

mros_df['submit_delay_sec'] = (mros_df['datetime_received_pacific'] - mros_df['createdtime']).dt.total_seconds()
mros_df['submit_delay_sec'].hist(bins=50)


In [ ]:
pip install seaborn

In [ ]:
# Rain vs Snow Distribution
import seaborn as sns
sns.countplot(data=mros_df, x='phase')


In [ ]:
# Rain vs Snow Distribution group by spatial regions:
mros_df.groupby(['geohash5', 'phase']).size().unstack().fillna(0)


In [ ]:
# Outlier Detection
# Invalid lat/lon values
mros_df['latitude'] = pd.to_numeric(mros_df['latitude'], errors='coerce')
mros_df['longitude'] = pd.to_numeric(mros_df['longitude'], errors='coerce')

lat_outliers = mros_df[(mros_df['latitude'] > 90) | (mros_df['latitude'] < -90)]
lon_outliers = mros_df[(mros_df['longitude'] > 180) | (mros_df['longitude'] < -180)]

lat_outliers

In [ ]:
# Time-of-Day Patterns
mros_df['hour'] = pd.to_datetime(mros_df['time_submitted_local'], errors='coerce').dt.hour
mros_df['hour'].value_counts().sort_index().plot(kind='bar', title="Submissions by Hour of Day")


## Stations Data (HADS, LCD & WCC)

In [ ]:
import pandas as pd

# Read the parquet file
hads_df = pd.read_csv(r'..\Data\Stations\hads_20241001_20250531.csv')
lcd_df = pd.read_csv(r'..\Data\Stations\lcd_20241001_20250531.csv')
wcc_df = pd.read_csv(r'..\Data\Stations\wcc_20241001_20250531.csv')

In [ ]:
hads_df

In [ ]:
lcd_df

In [ ]:
wcc_df